In [2]:
import pandas as pd

In [19]:
df = pd.read_csv('data/processed/dataset_sucio.csv')

In [20]:
# limpiamos los nulos, duplicados y columnas innecesarias

In [21]:
eliminar = []

for col in df.columns:
    if df[col].isna().sum() > len(df) * 0.15:
        eliminar.append(col)

print(eliminar)


['pobr', 'fum', 'alc', 'obes', 'fyv', 'sati']


In [22]:
df = df.drop(columns= eliminar)

Las variables 'crim' y 'tasa_criminalidad' son iguales... en 'crim' tenemos mas nulos por que no tenemos datos de Castilla Y León, esto seguramente se debe a una perdida de datos a la hora de formar el dataset.  
Las varibles 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler' faltan datos de algunos años de Ceuta, Melilla, Navarra, País Vasco y Rioja. No parece ser una perdida de datos si no a la no existencia de los mismos.

In [23]:
df = df.drop(columns= ['crim'])

In [24]:
col_con_nulos = ['homi','tasa_criminalidad', 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler'] 

In [25]:
for col in col_con_nulos:
    bfill = df.groupby("com_aut")[col].bfill()
    mask = (df["año"] == 2009) & (df[col].isna())
    df.loc[mask, col] = bfill[mask]

In [26]:
cols_precios = ["precio_medio_anual_eur_m2_venta", "precio_medio_anual_eur_m2_alquiler"]
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = (
    df.groupby("com_aut")[cols_precios]
      .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
)

In [27]:
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = df.groupby("com_aut")[cols_precios].ffill()

In [28]:
df[df.duplicated()]

,año,com_aut,pib,pob,sui,nat,paro,ing,homi,pib_pc,...,retrasos_pagos(%),renta_media,renta_mediana,riesgo_pobreza(%),dificultad_fin_mes(%),desigualdad_ing(S80/S20),inc_gastos_imprevistos(%),tasa_criminalidad,precio_medio_anual_eur_m2_venta,precio_medio_anual_eur_m2_alquiler


In [29]:
df.isna().sum()

año                                   0
com_aut                               0
pib                                   0
pob                                   0
sui                                   0
nat                                   0
paro                                  0
ing                                   0
homi                                  0
pib_pc                                0
renta_pc                              0
poblacion                             0
gasto_elevado_vivienda(%)             0
falta_espacio_vivienda(%)             0
retrasos_pagos(%)                     0
renta_media                           0
renta_mediana                         0
riesgo_pobreza(%)                     0
dificultad_fin_mes(%)                 0
desigualdad_ing(S80/S20)              0
inc_gastos_imprevistos(%)             0
tasa_criminalidad                     0
precio_medio_anual_eur_m2_venta       0
precio_medio_anual_eur_m2_alquiler    0
dtype: int64

In [30]:
df = df.drop(columns= ['pob','pib_pc'])

In [31]:
df['pib_pc']= df['pib']/df['poblacion']

In [32]:
df.columns

Index(['año', 'com_aut', 'pib', 'sui', 'nat', 'paro', 'ing', 'homi',
       'renta_pc', 'poblacion', 'gasto_elevado_vivienda(%)',
       'falta_espacio_vivienda(%)', 'retrasos_pagos(%)', 'renta_media',
       'renta_mediana', 'riesgo_pobreza(%)', 'dificultad_fin_mes(%)',
       'desigualdad_ing(S80/S20)', 'inc_gastos_imprevistos(%)',
       'tasa_criminalidad', 'precio_medio_anual_eur_m2_venta',
       'precio_medio_anual_eur_m2_alquiler', 'pib_pc'],
      dtype='object')

In [46]:
df = df.rename(columns = {'com_aut':'comunidad_autonoma',
                     'pib':'pbi_total',
                     'sui':'tasa_suicidio',
                     'nat':'tasa_natalidad',
                     'ing':'ingresos',
                     'homi':'tasa_homicidio',
                     'gasto_elevado_vivienda(%)': 'gasto_ele_vivienda(%)',
                     'falta_espacio_vivienda(%)': 'falta_espacio_vivienda(%)',
                     'precio_medio_anual_eur_m2_venta': 'precio_venta_eur_m2',
                     'precio_medio_anual_eur_m2_alquiler': 'precio_alq_eur_m2'
    
})

In [47]:
df = df[['año', 'comunidad_autonoma','poblacion', 'pbi_total','pib_pc', 'ingresos',
    'renta_pc',
    'renta_media', 'renta_mediana','tasa_natalidad', 'tasa_suicidio', 'tasa_homicidio', 
    'tasa_criminalidad','paro','gasto_ele_vivienda(%)', 'falta_espacio_vivienda(%)',
    'retrasos_pagos(%)','riesgo_pobreza(%)', 'dificultad_fin_mes(%)','desigualdad_ing(S80/S20)',
    'inc_gastos_imprevistos(%)','precio_venta_eur_m2', 'precio_alq_eur_m2'
       ]]
df.columns

Index(['año', 'comunidad_autonoma', 'poblacion', 'pbi_total', 'pib_pc',
       'ingresos', 'renta_pc', 'renta_media', 'renta_mediana',
       'tasa_natalidad', 'tasa_suicidio', 'tasa_homicidio',
       'tasa_criminalidad', 'paro', 'gasto_ele_vivienda(%)',
       'falta_espacio_vivienda(%)', 'retrasos_pagos(%)', 'riesgo_pobreza(%)',
       'dificultad_fin_mes(%)', 'desigualdad_ing(S80/S20)',
       'inc_gastos_imprevistos(%)', 'precio_venta_eur_m2',
       'precio_alq_eur_m2'],
      dtype='object')

In [13]:
df.to_csv("data/processed/dataset_final.csv", index=False)

### Resumen del proceso de limpieza del dataset

En primer lugar, se realizó un análisis de valores ausentes por columna. A partir de este análisis, se decidió eliminar aquellas columnas que presentaban más del 15 % de valores nulos respecto al total de filas del dataset. Para ello, se calculó el umbral multiplicando el número total de filas por 0,15, y se eliminaron todas las columnas cuyo número de valores ausentes superaba dicho umbral.

En una segunda etapa, se identificaron variables redundantes que representaban la misma información. En estos casos, se conservó una única variable y se eliminó la columna duplicada para evitar introducir ruido o duplicación innecesaria en el análisis.

Posteriormente, se abordó el tratamiento de variables que presentaban valores ausentes concentrados principalmente en el año inicial del dataset (2009). Dado que el conjunto de datos comienza en ese año y que para algunas comunidades no existían datos oficiales en 2009 pero sí en 2010, se optó por imputar dichos valores utilizando el dato correspondiente al año 2010 dentro de la misma comunidad autónoma, preservando así la continuidad temporal sin eliminar el año completo.

Por último, para las variables de precio medio anual de vivienda (venta y alquiler), se aplicó una estrategia de imputación en dos pasos. En primer lugar, se realizó una interpolación temporal lineal intra-comunidad para completar huecos intermedios entre años con datos disponibles. En segundo lugar, para los valores ausentes que permanecían en los años finales de la serie (al no existir un año posterior que permitiera la interpolación), se utilizó un forward fill intra-comunidad, imputando dichos valores con el último dato disponible para la misma comunidad. Este procedimiento se limitó exclusivamente a las variables de precios y se consideró una aproximación conservadora.



| Variable | Descripción | Tipo de variable | Importancia inicial |
|---------|-------------|------------------|---------------------|
| año | Año de referencia de los datos. | Numérica discreta | 0 |
| comunidad_autonoma | Comunidad Autónoma de España. | Categórica nominal | 0 |
| poblacion | Número total de habitantes por comunidad y año. | Numérica discreta | 3 |
| pbi_total | Producto Interior Bruto total. Valor económico agregado. | Numérica continua | 0 |
| pib_pc | PIB per cápita a precios de mercado. | Numérica continua | 1 |
| ingresos | Ingresos medios según la fuente del dataset. | Numérica continua | 1 |
| renta_pc | Renta disponible bruta de los hogares per cápita. | Numérica continua | 1 |
| renta_media | Renta media por unidad de consumo. | Numérica continua | 1 |
| renta_mediana | Renta mediana por unidad de consumo. | Numérica continua | 1 |
| tasa_natalidad | Nacimientos por cada 1.000 habitantes. | Numérica continua | 2 |
| tasa_suicidio | Defunciones por suicidio por 100.000 habitantes. | Numérica continua | 2 |
| tasa_homicidio | Homicidios y asesinatos por 100.000 habitantes. | Numérica continua | 3 |
| tasa_criminalidad | Delitos o infracciones penales por 1.000 habitantes. | Numérica continua | 2 |
| paro | Tasa de desempleo sobre población activa. | Numérica continua | 1 |
| gasto_ele_vivienda(%) | Población con gasto elevado en vivienda. | Numérica continua | 2 |
| falta_espacio_vivienda(%) | Población que vive en viviendas con falta de espacio. | Numérica continua | 3 |
| retrasos_pagos(%) | Población con retrasos en pagos de vivienda o suministros. | Numérica continua | 2 |
| riesgo_pobreza(%) | Población en riesgo de pobreza. | Numérica continua | 1 |
| dificultad_fin_mes(%) | Población con dificultad para llegar a fin de mes. | Numérica continua | 2 |
| desigualdad_ing(S80/S20) | Cociente entre el 20% más rico y el 20% más pobre. | Numérica continua | 1 |
| inc_gastos_imprevistos(%) | Población que no puede afrontar gastos imprevistos. | Numérica continua | 2 |
| precio_alq_eur_m2 | Precio medio anual de vivienda en alquiler por m² (€). | Numérica continua | 2 |
| precio_venta_eur_m2 | Precio medio anual de vivienda en venta por m² (€). | Numérica continua | 2 |
